# Run Video QA Streamlit App on Colab (GPU) via Cloudflare Tunnel

**Before running:** set the runtime to GPU --> *Runtime > Change runtime type > Hardware accelerator > GPU (T4)*.

Run the cells top-to-bottom. The last cell prints a public `https://*.trycloudflare.com` URL - open it to use the app.

> This variant loads **all** model artifacts (binary + multi-class) from a single Drive folder: `/content/drive/MyDrive/models3`.

## 1. Clone the repository

In [ ]:
import os

REPO_URL = "https://github.com/sudeeprana8043-svg/Streamlit_project.git"
REPO_DIR = "/content/Streamlit_project"

if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull

%cd $REPO_DIR
!ls

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 3. Configure model files

`MODEL_DIR` stays pointed at the repo's `model/` folder, which already ships the app-required legacy files (`binary_model.pkl`, `model_config.pkl`, `le_*.pkl`, `checkpoint-140/`). This cell then:

- copies every artifact from `/content/drive/MyDrive/models3` into `model/` (binary ensemble + multi-class heads),
- pulls `temporal_adapter.pt` (and `binary_model.pkl` / `model_config.pkl` if absent) from the old `/content/drive/MyDrive/models` folder,
- loads the **summarization checkpoint** `ucf_qwen_v9_qformer/checkpoint-414` into the `model/checkpoint-140` folder name the app expects,
- sets `MODEL_DIR` and `BINARY_MODEL_DIR` to the repo `model/` folder.

In [ ]:
import os, shutil

from google.colab import drive
drive.mount("/content/drive")

# The cloned repo's model/ folder ships the TEMPORAL-matched artifacts the app
# requires: binary_model.pkl, model_config.pkl, le_*.pkl, checkpoint-140/.
# These MUST stay in sync with temporal_adapter.pt, so we must NOT overwrite
# them with the multi-class encoders from models3 (different class counts ->
# "argmax of an empty sequence" in build_context_v2).
MODEL_LOCAL  = os.path.abspath("model")
os.makedirs(MODEL_LOCAL, exist_ok=True)

DRIVE_MODELS = "/content/drive/MyDrive/models3"

# Files in models3 that COLLIDE with the temporal-matched repo files.
# Keep the repo's originals -> do NOT copy these from models3.
SKIP = {
    "le_weapon.pkl", "le_location.pkl", "le_people.pkl", "le_super.pkl",
    "model_config.pkl", "binary_model.pkl", "model_metadata.pkl",
}

# 1) Copy models3 artifacts into model/, skipping the colliding encoders
copied, skipped = [], []
for fname in sorted(os.listdir(DRIVE_MODELS)):
    src = os.path.join(DRIVE_MODELS, fname)
    if not os.path.isfile(src):
        continue
    if fname in SKIP:
        skipped.append(fname)
        continue
    shutil.copy(src, os.path.join(MODEL_LOCAL, fname))
    copied.append(fname)
print(f"Copied {len(copied)} files from models3 -> model/")
print(f"Skipped (kept repo originals): {skipped}")

# 2) temporal_adapter.pt lives in the old Drive 'models' folder (not in models3)
OLD_DRIVE_MODELS = "/content/drive/MyDrive/models"
for legacy in ["temporal_adapter.pt", "binary_model.pkl", "model_config.pkl"]:
    dst = os.path.join(MODEL_LOCAL, legacy)
    src = os.path.join(OLD_DRIVE_MODELS, legacy)
    if not os.path.exists(dst) and os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Copied legacy {legacy} from {OLD_DRIVE_MODELS} -> model/")

# 3) Summarization LoRA: load checkpoint-414 weights into the checkpoint-140
#    folder name that the app expects.
SUMMARY_CKPT_DRIVE = "/content/drive/MyDrive/Project_VLM/ucf_qwen_v9_qformer/checkpoint-414"
SUMMARY_CKPT_LOCAL = os.path.join(MODEL_LOCAL, "checkpoint-140")
if not os.path.isdir(SUMMARY_CKPT_DRIVE):
    raise FileNotFoundError(f"Summary checkpoint not found in Drive: {SUMMARY_CKPT_DRIVE}")
shutil.copytree(SUMMARY_CKPT_DRIVE, SUMMARY_CKPT_LOCAL, dirs_exist_ok=True)
os.environ["SUMMARY_CHECKPOINT"] = SUMMARY_CKPT_LOCAL
os.environ["LORA_CHECKPOINT"] = SUMMARY_CKPT_LOCAL
print(f"Loaded checkpoint-414 weights into {SUMMARY_CKPT_LOCAL}")

# 4) Point the app at the repo model/ folder
os.environ["MODEL_DIR"] = MODEL_LOCAL
os.environ["BINARY_MODEL_DIR"] = MODEL_LOCAL

# ---- Sanity checks ----
required_model = [
    "binary_model.pkl", "model_config.pkl",
    "le_weapon.pkl", "le_location.pkl", "le_people.pkl", "le_super.pkl",
    "temporal_adapter.pt",
    "checkpoint-140/adapter_config.json",
    "checkpoint-140/adapter_model.safetensors",
]
missing_model = [f for f in required_model if not os.path.exists(os.path.join(MODEL_LOCAL, f))]
print("model/ missing (app-required):", missing_model if missing_model else "none")

required_binary = ["input_dim.pkl", "simple_adapter.pt", "lora_model.pt"]
missing_binary = [f for f in required_binary if not os.path.exists(os.path.join(MODEL_LOCAL, f))]
print("binary ensemble missing:", missing_binary if missing_binary else "none")

## 4. Download the Cloudflare tunnel binary

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!./cloudflared --version

## 5. Launch Streamlit + public tunnel

This starts Streamlit in the background, then opens a Cloudflare tunnel.
Watch the output for a line like `https://something.trycloudflare.com` and open it in a new tab.
Keep this cell running while you use the app.

In [ ]:
import subprocess, time, os, sys

PORT = 8501
APP = "streamlit_app_v2.py"  # v2 = SimpleAdapter + LoRA ensemble binary classifier

# Start Streamlit (headless) in the background
streamlit_proc = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", APP,
        "--server.port", str(PORT),
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
    ],
    stdout=open("/content/streamlit.log", "w"),
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)

print(f"Starting {APP}... (give it ~20s to boot and load models)")
time.sleep(20)
print("Recent Streamlit log:")
!tail -n 20 /content/streamlit.log

# Open the public tunnel (this blocks and prints the trycloudflare URL)
print("\n=== Public URL will appear below (look for *.trycloudflare.com) ===\n")
!./cloudflared tunnel --url http://localhost:$PORT